In [7]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig 
from dynclim.experiments.runner import ExperimentRunner
from dynclim.data import NRCDataPipeline
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
# project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")
project_root = Path("/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit")



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/safety_domains.yaml')

# Get data for feature extraction
# pipeline = NRCDataPipeline(output_dir=project_root / "dynclim/data/processed/nrc")
# events_df = pipeline.scrape_events(project_root / "dynclim/data/raw/nrc/ENSearchResults_02-16-2026.xlsx")

# clean and prep reports for feature generation
pipeline = NRCDataPipeline(
    events_path=project_root / "dynclim/data/raw/nrc/nrc_events_raw.csv",
    output_dir=project_root / "dynclim/data/processed/nrc"
)
results = pipeline.run()
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]

2026-02-17 08:33:10,491 - INFO - ============================================================
2026-02-17 08:33:10,491 - INFO - Running NRC Data Pipeline
2026-02-17 08:33:10,492 - INFO - ============================================================
2026-02-17 08:33:10,492 - INFO - Loading NRC events from /Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/raw/nrc/nrc_events_raw.csv
2026-02-17 08:33:10,642 - INFO - Loaded 14,110 raw event records
2026-02-17 08:33:10,970 - INFO - Cleaning: 14,110 -> 14,110 events (0 removed)
2026-02-17 08:33:11,234 - INFO - Saved 14,110 records to /Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/nrc/events.parquet
2026-02-17 08:33:11,234 - INFO - ============================================================
2026-02-17 08:33:11,234 - INFO - Pipeline complete
2026-02-17 08:33:11,234 - INFO - ============================================================


In [ ]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[      # first three are from same model family but differ in size
                                 # last option is more performant in general, but a different model family
        'all-MiniLM-L6-v2',      # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        'BAAI/bge-large-en-v1.5' # very large, and generally better performance, but has different training method
                                 # when compared to above models which vary just on size
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['roberta', 'distilbert'] #, 'finbert','vader']
}

results_df = runner.run_experiment(
    report_data, 
    experiment_name = "nrc_full_run_02-17-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)

   Registry: dropped 21 constant columns → 13 remaining (constants in registry_constants.json)

🔬 Running experiment 'nrc_full_run_02-17-2026'
   Total configurations: 540
   Pending configurations: 540
   Reports: 14110
   Embedding models: 3
   Parallel processing: False
   Caching: Enabled (embeddings=True, segments=True)


Configurations:   0%|          | 0/540 [00:00<?, ?it/s]


🔄 Initializing processor:
   Embedding model: BAAI/bge-large-en-v1.5
   Sentiment method: distilbert
🚀 Using Apple Silicon MPS acceleration


2026-02-17 08:41:46,645 - INFO - Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Initialized Processor:
   Domains: ['LSC', 'SC', 'ST', 'CSP', 'SEH', 'SI', 'SR']
   Total items: 30
   Device: mps
   Caching: embeddings=True, segments=True
   Injected sentiment cache: 0 entries

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
[seg] SentenceSegmenter backend=spacy, min_length=5, unit=chars
[seg] built segmenter SentenceSegmenter for key=('5a5a91d711baaba23988779567c0b9f2', 'chars')
   Found 126254 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 126254 embeddings

🚀 Starting processing...
   Reports: 14110
   Parallel: False


... (479 outputs left out)

Configurations:  44%|████▍     | 237/540 [11:28:50<13:47:02, 163.77s/it, cache_hit=99.4%, time=179.5s]

✅ Processing complete!
   Total time: 178.86 seconds
   Rate: 78.9 reports/second

💾 Checkpoint saved: 237 configurations
   Full results directory: checkpoints/nrc_full_run_02-17-2026/_cfg

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
   Found 125750 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 125750 embeddings

🚀 Starting processing...
   Reports: 14110
   Parallel: False


Configurations:  44%|████▍     | 238/540 [11:31:51<14:10:38, 169.00s/it, cache_hit=99.4%, time=181.1s]

✅ Processing complete!
   Total time: 180.48 seconds
   Rate: 78.2 reports/second

💾 Checkpoint saved: 238 configurations
   Full results directory: checkpoints/nrc_full_run_02-17-2026/_cfg

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
   Found 125750 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 125750 embeddings

🚀 Starting processing...
   Reports: 14110
   Parallel: False
